# 🧠 Model-Based RL using MCTS (Simple Explanation)

## 📌 What is this?
This project uses **Monte Carlo Tree Search (MCTS)** to make better decisions in a Reinforcement Learning environment.

---

## 🎯 Goal
Balance the pole in **CartPole-v1** and get maximum reward.

---

## 🎮 Actions
0 = LEFT
1 = RIGHT


---

## 🧩 Key Idea

Instead of acting randomly, the agent:
👉 **thinks ahead by simulating future actions**

---

## 🌳 How MCTS Works (4 Steps)

### 1. Selection
Choose the best action using a formula (UCB)

---

### 2. Expansion
Add a new possible action to the tree

---

### 3. Simulation
Play randomly to see what reward we get

---

### 4. Backpropagation
Update the results back to the tree

---

## 🔁 Loop

For every step:
1. Run MCTS simulations
2. Pick best action
3. Perform action
4. Repeat

---

## ⚙️ Environment

- State → 4 values (position, velocity, angle, angular velocity)
- Actions → Left / Right
- Reward → +1 for every step alive

---

## 📊 Output
Step 1 | Action: 1 | Reward: 1
Step 2 | Action: 0 | Reward: 1
...
Total Reward: 45

## ✅ Conclusion
- MCTS = planning before acting  
- Helps choose better actions  
- Improves performance over random decisions  

In [14]:
import gymnasium as gym
import numpy as np
import random
import copy

In [16]:
# Create CartPole environment
# CartPole is simple → good for testing MCTS
env = gym.make("CartPole-v1")

# Reset environment to get initial state
# Node represents a state in the MCTS tree
state, _ = env.reset()

In [19]:
# Node represents a state in the MCTS tree
class Node:
    def __init__(self, state, parent=None):
        self.state = state        # current state
        self.parent = parent      # parent node
        self.children = {}        # dictionary: action -> child node
        self.visits = 0           # number of times node visited
        self.value = 0            # total reward accumulated

    # Check if all possible actions are explored
    def is_fully_expanded(self, action_space):
        return len(self.children) == action_space.n


## UCB Function (Selection Strate

In [21]:
# Upper Confidence Bound (UCB)
# Balances:
# - Exploitation (high value)
# - Exploration (low visits)
def ucb(parent, child, c=1.4):
    if child.visits == 0:
        return float('inf')  # force exploration

    return (child.value / child.visits) + c * np.sqrt(np.log(parent.visits + 1) / child.visits)

## MCTS Algorithm

In [23]:
# Monte Carlo Tree Search Class
class MCTS:
    def __init__(self, env, simulations=20, max_depth=50):
        self.env = env
        self.simulations = simulations   # number of simulations per step
        self.max_depth = max_depth       # limit rollout depth

    def search(self, root):
        # Run multiple simulations
        for _ in range(self.simulations):

            # 🔥 IMPORTANT: Clone environment
            # So simulation starts from CURRENT state
            sim_env = copy.deepcopy(self.env)

            node = root
            done = False
            total_reward = 0
            depth = 0

            # =========================
            # 1. SELECTION
            # =========================
            # Traverse tree using best UCB score
            while node.children and not done:
                action, node = max(
                    node.children.items(),
                    key=lambda item: ucb(node, item[1])
                )

                _, reward, terminated, truncated, _ = sim_env.step(action)
                done = terminated or truncated
                total_reward += reward
                depth += 1

            # =========================
            # 2. EXPANSION
            # =========================
            # Add new child if not fully expanded
            if not done and not node.is_fully_expanded(sim_env.action_space):
                untried = [
                    a for a in range(sim_env.action_space.n)
                    if a not in node.children
                ]

                action = random.choice(untried)

                next_state, reward, terminated, truncated, _ = sim_env.step(action)
                done = terminated or truncated
                total_reward += reward

                child = Node(next_state, parent=node)
                node.children[action] = child
                node = child

            # =========================
            # 3. SIMULATION (ROLLOUT)
            # =========================
            # Play randomly until terminal state
            while not done and depth < self.max_depth:
                action = sim_env.action_space.sample()
                _, reward, terminated, truncated, _ = sim_env.step(action)
                done = terminated or truncated
                total_reward += reward
                depth += 1

            # =========================
            # 4. BACKPROPAGATION
            # =========================
            # Update values back to root
            while node:
                node.visits += 1
                node.value += total_reward
                node = node.parent

    # Select best action based on visits
    def best_action(self, root):
        return max(root.children.items(), key=lambda x: x[1].visits)[0]

## Test MCTS

In [24]:
# Initialize MCTS
mcts = MCTS(env, simulations=10)

# Create root node
root = Node(state)

# Run MCTS search
mcts.search(root)

print("✅ MCTS ran successfully")
print("Available actions:", list(root.children.keys()))

# Show statistics for each action
for action, child in root.children.items():
    print(f"Action {action} -> Visits: {child.visits}, Value: {child.value}")

✅ MCTS ran successfully
Available actions: [0]
Action 0 -> Visits: 10, Value: 220.0


In [25]:
episodes = 2

for ep in range(episodes):
    state, _ = env.reset()
    root = Node(state)

    done = False
    total_reward = 0
    step = 0

    print(f"\n🚀 Episode {ep+1} started")

    while not done:
        step += 1

        # Run MCTS to decide best action
        mcts.search(root)
        action = mcts.best_action(root)

        # Take action in real environment
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward

        # Print step info
        print(f"Step {step} | Action: {action} | Reward: {reward}")

        # Move tree forward
        if action in root.children:
            root = root.children[action]
            root.parent = None
        else:
            root = Node(next_state)

    # Episode summary
    print("===================================")
    print(f"✅ Episode {ep+1} Finished")
    print(f"🎯 Total Reward: {total_reward}")
    print("===================================")


🚀 Episode 1 started
Step 1 | Action: 1 | Reward: 1.0
Step 2 | Action: 1 | Reward: 1.0
Step 3 | Action: 0 | Reward: 1.0
Step 4 | Action: 1 | Reward: 1.0
Step 5 | Action: 1 | Reward: 1.0
Step 6 | Action: 1 | Reward: 1.0
Step 7 | Action: 1 | Reward: 1.0
Step 8 | Action: 1 | Reward: 1.0
Step 9 | Action: 0 | Reward: 1.0
Step 10 | Action: 0 | Reward: 1.0
Step 11 | Action: 1 | Reward: 1.0
Step 12 | Action: 0 | Reward: 1.0
Step 13 | Action: 0 | Reward: 1.0
✅ Episode 1 Finished
🎯 Total Reward: 13.0

🚀 Episode 2 started
Step 1 | Action: 1 | Reward: 1.0
Step 2 | Action: 1 | Reward: 1.0
Step 3 | Action: 1 | Reward: 1.0
Step 4 | Action: 0 | Reward: 1.0
Step 5 | Action: 1 | Reward: 1.0
Step 6 | Action: 1 | Reward: 1.0
Step 7 | Action: 0 | Reward: 1.0
Step 8 | Action: 1 | Reward: 1.0
Step 9 | Action: 1 | Reward: 1.0
Step 10 | Action: 1 | Reward: 1.0
Step 11 | Action: 1 | Reward: 1.0
Step 12 | Action: 0 | Reward: 1.0
✅ Episode 2 Finished
🎯 Total Reward: 12.0
